In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

print("Entorno listo para JARVIS")

# Proyecto de Minería de Datos – Grupo JARVIS

## Predicción del Riesgo de Deserción Estudiantil en Educación Superior

**Metodología:** CRISP-DM  
**Fase:** Comprensión del Negocio y Comprensión de los Datos  
**Dataset principal:** Open University Learning Analytics Dataset (OULAD)

## 1. Carga y descripción del dataset

In [ ]:
url = "https://raw.githubusercontent.com/marloft/MachineLearning/refs/heads/master/Documents/ML/PhD/Datasets/Open%20University%20Learning%20Analytics%20Dataset%20-%20OULAD/studentInfo.csv"

df = pd.read_csv(url)

df.head()

In [ ]:
print("Número de registros:", df.shape[0])
print("Número de variables:", df.shape[1])

print("\nColumnas del dataset:")
print(df.columns.tolist())

In [ ]:
df.info()

### Ficha técnica del dataset

- **Registros:** 32.593
- **Variables originales:** 12
- **Unidad de análisis:** inscripción estudiante–módulo–presentación.
- **Variables numéricas de análisis:** `num_of_prev_attempts`, `studied_credits`.
- **Identificador:** `id_student`.
- **Variables categóricas:** `code_module`, `code_presentation`, `gender`, `region`,
  `highest_education`, `imd_band`, `age_band`, `disability`, `final_result`.
- **Variable objetivo derivada:** `abandono`.

## 2. Definición de la variable objetivo

In [ ]:
df["abandono"] = (df["final_result"] == "Withdrawn").astype(int)

df[["final_result", "abandono"]].head(10)

### Definición del target

La variable objetivo `abandono` se deriva de `final_result`:

- `abandono = 1`: `Withdrawn`.
- `abandono = 0`: `Pass`, `Fail` o `Distinction`.

`final_result` se utiliza únicamente para construir la etiqueta y deberá excluirse
como predictor durante el modelado para evitar fuga de información. `id_student`
tampoco será utilizado como predictor.

In [ ]:
print(df["abandono"].value_counts())

In [ ]:
print(df["abandono"].value_counts(normalize=True) * 100)

## 3. Calidad de los datos

In [ ]:
print(df.isnull().sum())

In [ ]:
total = len(df)
abandonos = df["abandono"].sum()
no_abandonos = total - abandonos
tasa_abandono = df["abandono"].mean()

print("=== BASELINE DE DESERCIÓN ===")
print("Total de registros:", total)
print("Abandonos:", abandonos)
print("No abandonos:", no_abandonos)
print(f"Tasa de abandono: {tasa_abandono:.2%}")

In [ ]:
duplicados = df.duplicated().sum()
estudiantes_unicos = df["id_student"].nunique()

print("Registros duplicados exactos:", duplicados)
print("Total de registros:", len(df))
print("Estudiantes únicos:", estudiantes_unicos)

In [ ]:
faltantes_imd = df["imd_band"].isnull().sum()
porcentaje_imd = df["imd_band"].isnull().mean() * 100

print("Faltantes en imd_band:", faltantes_imd)
print(f"Porcentaje de faltantes: {porcentaje_imd:.2f}%")

## 4. Cálculo del baseline y meta

In [ ]:
reduccion_objetivo = 0.10
meta_abandono = tasa_abandono * (1 - reduccion_objetivo)

print(f"Baseline actual: {tasa_abandono:.2%}")
print(f"Reducción propuesta: {reduccion_objetivo:.0%}")
print(f"Meta de tasa de abandono: {meta_abandono:.2%}")

## 5. Análisis Exploratorio de Datos (EDA)

In [ ]:
conteo_abandono = df["abandono"].value_counts().sort_index()

plt.figure(figsize=(7, 5))

barras = plt.bar(
    ["No abandono", "Abandono"],
    [conteo_abandono[0], conteo_abandono[1]]
)

plt.title("Distribución de la variable objetivo: abandono")
plt.ylabel("Número de registros")
plt.xlabel("Condición")

for barra in barras:
    plt.text(
        barra.get_x() + barra.get_width()/2,
        barra.get_height() + 200,
        f"{int(barra.get_height()):,}",
        ha="center"
    )

plt.show()

### Interpretación de la variable objetivo

El dataset contiene 10.156 casos de abandono y 22.437 casos de no abandono. 
Esto representa una tasa histórica de abandono del 31,16 %, valor que se 
establece como línea base (baseline) del proyecto.

La distribución presenta un desbalance moderado entre ambas clases, por lo que 
en etapas posteriores será necesario considerar métricas de evaluación como 
Recall y F1-score, además de la exactitud.

In [ ]:
abandono_modulo = (
    df.groupby("code_module")["abandono"]
      .mean()
      .sort_values(ascending=False) * 100
)

print(abandono_modulo.round(2))

In [ ]:
plt.figure(figsize=(8, 5))

abandono_modulo.plot(kind="bar")

plt.title("Tasa de abandono por módulo")
plt.xlabel("Módulo")
plt.ylabel("Tasa de abandono (%)")
plt.xticks(rotation=0)

plt.show()

### Interpretación del abandono por módulo

La tasa de abandono presenta diferencias importantes entre los módulos. Esto 
indica que el riesgo de abandono no es uniforme y que el módulo cursado puede 
estar relacionado con la probabilidad de retiro.

Por tanto, `code_module` constituye una variable relevante para continuar 
analizando durante las siguientes fases del proyecto.

In [ ]:
df["studied_credits"].describe()

In [ ]:
plt.figure(figsize=(9, 4))

plt.boxplot(df["studied_credits"], orientation="horizontal")

plt.title("Distribución y posibles valores atípicos de studied_credits")
plt.xlabel("Créditos estudiados")

plt.show()

### Análisis de posibles valores atípicos

La variable `studied_credits` presenta valores elevados respecto a la mayoría 
de los registros. Estos valores son considerados inicialmente como posibles 
outliers.

Sin embargo, durante el EDA no se eliminarán automáticamente, ya que primero 
debe determinarse si corresponden a errores de registro o a estudiantes con una 
carga académica excepcionalmente alta.

In [ ]:
promedio_creditos = df.groupby("abandono")["studied_credits"].mean()

print("Promedio de créditos:")
print(promedio_creditos)

In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(
    ["No abandono", "Abandono"],
    [promedio_creditos[0], promedio_creditos[1]]
)

plt.title("Promedio de créditos según condición de abandono")
plt.ylabel("Promedio de créditos estudiados")
plt.xlabel("Condición")

plt.show()

In [ ]:
df["rango_creditos"] = pd.cut(
    df["studied_credits"],
    bins=[0, 60, 120, 180, float("inf")],
    labels=["Hasta 60", "61-120", "121-180", "Más de 180"]
)

abandono_creditos = (
    df.groupby("rango_creditos", observed=True)["abandono"]
      .mean() * 100
)

print(abandono_creditos.round(2))

In [ ]:
plt.figure(figsize=(8, 5))

abandono_creditos.plot(kind="bar")

plt.title("Tasa de abandono según carga de créditos")
plt.xlabel("Créditos estudiados")
plt.ylabel("Tasa de abandono (%)")
plt.xticks(rotation=0)

plt.show()

## 6. Hipótesis inicial

A partir del análisis exploratorio se plantea que características académicas 
como el módulo cursado y la carga de créditos pueden estar relacionadas con 
la probabilidad de abandono estudiantil.

La hipótesis preliminar es que determinados contextos académicos y mayores 
cargas de estudio presentan tasas de abandono diferentes respecto al resto 
de los estudiantes.

Estas relaciones son exploratorias y no implican causalidad. Su capacidad 
predictiva deberá comprobarse posteriormente durante las fases de preparación 
de datos, modelado y evaluación de CRISP-DM.